In [1]:
import pandas as pd
import numpy as np

In [2]:
nvidia = pd.read_csv('nvidia_with_features.csv')

### Creating necessary features for classification

In [3]:
nvidia['MACD_Histogram'] = nvidia['MACD'] - nvidia['Signal_Line']

In [4]:
nvidia['Price_MA20_Diff'] = (
    (nvidia['Close'] - nvidia['MA20']) / nvidia['MA20']
)

In [5]:
nvidia['Price_MA50_Diff'] = (
    (nvidia['Close'] - nvidia['MA50']) / nvidia['MA50']
)

In [6]:
nvidia['RSI_Change'] = nvidia['RSI'].diff()

In [7]:
nvidia['Volatility_Ratio'] = (
    nvidia['Volatility'] /
    nvidia['Volatility'].rolling(20).mean()
)

In [8]:
nvidia['Volume_Ratio'] = (
    nvidia['Volume'] /
    nvidia['Volume_MA20']
)

In [9]:
nvidia['MA20_MA50_Ratio'] = (
    nvidia['MA20'] / nvidia['MA50']
)

## LSTM Classification

In [10]:
nvidia['Target'] = (
    nvidia['Close'].shift(-1) > nvidia['Close']
).astype(int)

In [11]:
features = [
    'Returns',
    'RSI',
    'MACD',
    'Signal_Line',
    'MACD_Histogram',
    'Momentum',
    'Volatility',
    'BB_Width',
    'Volume_Change',
    'Volume_Ratio',
    'Lag_R1',
    'Lag_R2',
    'Lag_1',
    'Lag_2',
    'Lag_3',
    'Lag_5',
    'Price_MA20_Diff',
    'Price_MA50_Diff',
    'MA20_MA50_Ratio'
]

In [12]:
nvidia.dropna(inplace=True)

In [13]:
split = int(len(nvidia) * 0.8)

train_data = nvidia.iloc[:split]
test_data = nvidia.iloc[split:]

In [14]:
from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(
    train_data[features]
)

X_test_scaled = feature_scaler.transform(
    test_data[features]
)

In [15]:
y_train = train_data['Target'].values
y_test = test_data['Target'].values

In [16]:
sequence_length = 60

X_train = []
y_train_seq = []

for i in range(sequence_length, len(X_train_scaled)):
    X_train.append(
        X_train_scaled[i-sequence_length:i]
    )

    y_train_seq.append(
        y_train[i]
    )

X_train = np.array(X_train)
y_train_seq = np.array(y_train_seq)

In [17]:
X_test = []
y_test_seq = []

for i in range(sequence_length, len(X_test_scaled)):
    X_test.append(
        X_test_scaled[i-sequence_length:i]
    )

    y_test_seq.append(
        y_test[i]
    )

X_test = np.array(X_test)
y_test_seq = np.array(y_test_seq)

In [18]:
print(X_train.shape)
print(y_train_seq.shape)

(5320, 60, 19)
(5320,)


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(
    LSTM(
        64,
        return_sequences=True,
        input_shape=(60, 19)
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(32)
)

model.add(Dropout(0.2))

model.add(
    Dense(
        1,
        activation='sigmoid'
    )
)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

d:\Anaconda\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 64)         │        21,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,953 (132.63 KB)

 Trainable params: 33,953 (132.63 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train_seq)

weights = compute_class_weight(
    'balanced',
    classes=classes,
    y=y_train_seq
)

class_weights = dict(zip(classes, weights))

model.fit(
    X_train,
    y_train_seq,
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    validation_data=(X_test, y_test_seq)
)

Epoch 1/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 29s 56ms/step - accuracy: 0.5102 - loss: 0.6944 - val_accuracy: 0.5354 - val_loss: 0.6930
Epoch 2/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.4942 - loss: 0.6937 - val_accuracy: 0.5354 - val_loss: 0.6945
Epoch 3/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.5034 - loss: 0.6935 - val_accuracy: 0.5354 - val_loss: 0.6976
Epoch 4/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.5038 - loss: 0.6930 - val_accuracy: 0.5354 - val_loss: 0.7120
Epoch 5/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step - accuracy: 0.5122 - loss: 0.6928 - val_accuracy: 0.5354 - val_loss: 0.7079
Epoch 6/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.4979 - loss: 0.6936 - val_accuracy: 0.5354 - val_loss: 0.7100
Epoch 7/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 6s 38ms/step - accuracy: 0.5165 - loss: 0.6927 - val_accuracy: 0.5354 - val_loss: 0.7012
Epoch 8/20
167/167 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.5100 - loss: 0.6929 - val_ac

In [22]:
y_pred_prob = model.predict(X_test)

y_pred = (
    y_pred_prob > 0.5
).astype(int)

41/41 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step


In [23]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print(
    accuracy_score(
        y_test_seq,
        y_pred
    )
)

print(
    confusion_matrix(
        y_test_seq,
        y_pred
    )
)

print(
    classification_report(
        y_test_seq,
        y_pred
    )
)

0.5291828793774319
[[ 31 566]
 [ 39 649]]
              precision    recall  f1-score   support

           0       0.44      0.05      0.09       597
           1       0.53      0.94      0.68       688

    accuracy                           0.53      1285
   macro avg       0.49      0.50      0.39      1285
weighted avg       0.49      0.53      0.41      1285

